# Day 2 — Feature Engineering & First XGBoost Model
## Objective
Build and evaluate our first XGBoost regression model
to predict delay_ratio.
## Baseline
Lane median with global median fallback.
Test MAE: 0.7405

In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
import sklearn
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("XGBoost version:", xgb.__version__)
print("Scikit-learn version:", sklearn.__version__)

Pandas version: 2.3.3
NumPy version: 2.2.6
XGBoost version: 3.2.0
Scikit-learn version: 1.7.2


In [8]:
from pathlib import Path

SPLIT = Path("../data/datasplit")

tcols = ["od_start_time", "trip_creation_time"]

train = pd.read_csv(
    SPLIT / "train_legs_v1.csv",
    parse_dates=tcols
)

cal = pd.read_csv(
    SPLIT / "calibration_legs_v1.csv",
    parse_dates=tcols
)

test = pd.read_csv(
    SPLIT / "test_legs_v1.csv",
    parse_dates=tcols
)

for name, part in [
    ("Training", train),
    ("Calibration", cal),
    ("Test", test)
]:
    print(
        name,
        "| Legs:", len(part),
        "| Trips:", part["trip_uuid"].nunique()
    )

Training | Legs: 15621 | Trips: 8890
Calibration | Legs: 5441 | Trips: 2963
Test | Legs: 5307 | Trips: 2964


In [9]:
train["route_type"].value_counts(dropna=False)

route_type
FTL        8468
Carting    7153
Name: count, dtype: int64

In [10]:
def build_features(df):
    start = pd.to_datetime(df["od_start_time"])

    X = pd.DataFrame(index=df.index)

    X["osrm_time_min"] = df["osrm_time_min"]
    X["osrm_distance_km"] = df["osrm_distance_km"]
    X["osrm_speed_kmh"] = (
        df["osrm_distance_km"] / (df["osrm_time_min"] / 60)
    )
    X["log_osrm_time"] = np.log1p(df["osrm_time_min"])

    X["start_hour"] = start.dt.hour
    X["start_dow"] = start.dt.dayofweek
    X["is_weekend"] = (start.dt.dayofweek >= 5).astype(int)

    X["is_ftl"] = (df["route_type"] == "FTL").astype(int)

    return X

In [11]:
X_train = build_features(train)
X_cal = build_features(cal)
X_test = build_features(test)

print("Training features:", X_train.shape)
print("Calibration features:", X_cal.shape)
print("Test features:", X_test.shape)

print("\nMissing values in training features:")
print(X_train.isnull().sum())

print("\nFeature summary:")
display(X_train.describe())

Training features: (15621, 8)
Calibration features: (5441, 8)
Test features: (5307, 8)

Missing values in training features:
osrm_time_min       0
osrm_distance_km    0
osrm_speed_kmh      0
log_osrm_time       0
start_hour          0
start_dow           0
is_weekend          0
is_ftl              0
dtype: int64

Feature summary:


,osrm_time_min,osrm_distance_km,osrm_speed_kmh,log_osrm_time,start_hour,start_dow,is_weekend,is_ftl
count,15621.000000,15621.000000,15621.000000,15621.000000,15621.000000,15621.000000,15621.000000,15621.000000
mean,93.038218,117.475676,70.415932,3.916501,8.966647,3.224057,0.315025,0.542091
std,183.444522,250.760704,11.695310,0.937333,7.444707,1.924866,0.464541,0.498241
min,6.000000,9.072900,25.862028,1.945910,0.000000,0.000000,0.000000,0.000000
25%,26.000000,28.481700,62.337195,3.295837,3.000000,2.000000,0.000000,0.000000
50%,41.000000,44.826300,71.335895,3.737670,6.000000,3.000000,0.000000,1.000000
75%,77.000000,91.613900,80.956533,4.356709,16.000000,5.000000,1.000000,1.000000
max,1686.000000,2326.199100,96.907000,7.430707,23.000000,6.000000,1.000000,1.000000


In [12]:
from sklearn.metrics import mean_absolute_error, median_absolute_error
import numpy as np

def evaluate_model(y_true, y_pred, osrm_time):
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)

    excess_true = (y_true - 1) * osrm_time
    excess_pred = (y_pred - 1) * osrm_time

    excess_mae = mean_absolute_error(excess_true, excess_pred)

    return {
        "MAE_ratio": round(mae, 4),
        "MedAE_ratio": round(medae, 4),
        "MAE_excess_min": round(excess_mae, 2)
    }

In [13]:
y_train = train["delay_ratio"]
y_cal = cal["delay_ratio"]
y_test = test["delay_ratio"]

global_median = y_train.median()

lane_medians = train.groupby(
    ["source_center", "destination_center"]
)["delay_ratio"].median()

def predict_lane_median(df):
    lane_keys = pd.Series(
        list(zip(df["source_center"], df["destination_center"])),
        index=df.index
    )

    predictions = lane_keys.map(
        dict(zip(lane_medians.index, lane_medians.values))
    )

    return predictions.fillna(global_median).to_numpy()

cal_global_pred = np.full(len(cal), global_median)
test_global_pred = np.full(len(test), global_median)

cal_lane_pred = predict_lane_median(cal)
test_lane_pred = predict_lane_median(test)

baseline_results = {
    "Global Median - Calibration": evaluate_model(
        y_cal, cal_global_pred, cal["osrm_time_min"]
    ),
    "Lane Median - Calibration": evaluate_model(
        y_cal, cal_lane_pred, cal["osrm_time_min"]
    ),
    "Global Median - Test": evaluate_model(
        y_test, test_global_pred, test["osrm_time_min"]
    ),
    "Lane Median - Test": evaluate_model(
        y_test, test_lane_pred, test["osrm_time_min"]
    )
}

baseline_results

{'Global Median - Calibration': {'MAE_ratio': 0.9648,
  'MedAE_ratio': 0.436,
  'MAE_excess_min': 47.37},
 'Lane Median - Calibration': {'MAE_ratio': 0.6395,
  'MedAE_ratio': 0.2476,
  'MAE_excess_min': 29.32},
 'Global Median - Test': {'MAE_ratio': 1.015,
  'MedAE_ratio': 0.4483,
  'MAE_excess_min': 49.31},
 'Lane Median - Test': {'MAE_ratio': 0.7406,
  'MedAE_ratio': 0.2851,
  'MAE_excess_min': 34.27}}

In [14]:
train_trip_dates = (
    train.groupby("trip_uuid")["trip_creation_time"]
    .min()
    .sort_values()
)

split_index = int(len(train_trip_dates) * 0.8)

fit_trip_ids = train_trip_dates.index[:split_index]
val_trip_ids = train_trip_dates.index[split_index:]

fit_df = train[train["trip_uuid"].isin(fit_trip_ids)].copy()
val_df = train[train["trip_uuid"].isin(val_trip_ids)].copy()

print("Fit trips:", fit_df["trip_uuid"].nunique())
print("Validation trips:", val_df["trip_uuid"].nunique())

print("Fit legs:", len(fit_df))
print("Validation legs:", len(val_df))

print("Fit end:", fit_df["trip_creation_time"].max())
print("Validation start:", val_df["trip_creation_time"].min())

Fit trips: 7112
Validation trips: 1778
Fit legs: 12514
Validation legs: 3107
Fit end: 2018-09-21 21:45:36.766749
Validation start: 2018-09-21 21:46:36.360521


In [15]:
X_fit = build_features(fit_df)
X_val = build_features(val_df)

y_fit = fit_df["delay_ratio"]
y_val = val_df["delay_ratio"]

print("X_fit:", X_fit.shape)
print("X_val:", X_val.shape)

print("Missing values in X_fit:", X_fit.isnull().sum().sum())
print("Missing values in X_val:", X_val.isnull().sum().sum())

X_fit: (12514, 8)
X_val: (3107, 8)
Missing values in X_fit: 0
Missing values in X_val: 0


In [16]:
from xgboost import XGBRegressor

xgb_v0 = XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    tree_method="hist",
    random_state=42,
    eval_metric="mae",
    early_stopping_rounds=50
)

xgb_v0.fit(
    X_fit,
    y_fit,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Best iteration:", xgb_v0.best_iteration)
print("Best validation MAE:", xgb_v0.best_score)

Best iteration: 1438
Best validation MAE: 0.8981125828994981


In [17]:
v0_val_pred = xgb_v0.predict(X_val)

v0_val_results = evaluate_model(
    y_val,
    v0_val_pred,
    val_df["osrm_time_min"]
)

v0_val_results

{'MAE_ratio': 0.8981, 'MedAE_ratio': 0.361, 'MAE_excess_min': 45.29}

In [18]:
val_lane_pred = predict_lane_median(val_df)

val_lane_results = evaluate_model(
    y_val,
    val_lane_pred,
    val_df["osrm_time_min"]
)

print("V0 XGBoost:", v0_val_results)
print("Lane Median:", val_lane_results)

V0 XGBoost: {'MAE_ratio': 0.8981, 'MedAE_ratio': 0.361, 'MAE_excess_min': 45.29}
Lane Median: {'MAE_ratio': 0.5662, 'MedAE_ratio': 0.1667, 'MAE_excess_min': 27.56}


In [19]:
import json
from pathlib import Path
from datetime import datetime, timezone

experiment_dir = Path("../experiments/day2")
experiment_dir.mkdir(parents=True, exist_ok=True)

v0_record = {
    "experiment": "V0",
    "description": "Initial XGBoost with 8 engineered features",
    "features": list(X_fit.columns),
    "fit_legs": len(fit_df),
    "validation_legs": len(val_df),
    "best_iteration": int(xgb_v0.best_iteration),
    "best_validation_mae": float(xgb_v0.best_score),
    "xgboost_validation_metrics": v0_val_results,
    "lane_median_validation_metrics": val_lane_results,
    "model_parameters": xgb_v0.get_params(),
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}
save_path = experiment_dir / "v0_results_v2.json"

if save_path.exists():
    raise FileExistsError(f"{save_path} already exists. Choose a new version.")

with open(save_path, "w") as f:
    json.dump(v0_record, f, indent=4, default=str)

print("Saved:", save_path)

FileExistsError: ..\experiments\day2\v0_results_v2.json already exists. Choose a new version.

In [20]:
print("Unique source hubs:", train["source_center"].nunique())
print("Unique destination hubs:", train["destination_center"].nunique())

print("\nMissing source hubs:", train["source_center"].isnull().sum())
print("Missing destination hubs:", train["destination_center"].isnull().sum())

print("\nSample source hubs:")
print(train["source_center"].unique()[:10])

print("\nSample destination hubs:")
print(train["destination_center"].unique()[:10])

Unique source hubs: 1388
Unique destination hubs: 1367

Missing source hubs: 0
Missing destination hubs: 0

Sample source hubs:
['IND209304AAA' 'IND462022AAA' 'IND561203AAB' 'IND572101AAA'
 'IND000000ACB' 'IND562132AAA' 'IND400072AAB' 'IND583101AAA'
 'IND583119AAA' 'IND583201AAA']

Sample destination hubs:
['IND000000ACB' 'IND209304AAA' 'IND562101AAA' 'IND561203AAB'
 'IND160002AAC' 'IND401104AAA' 'IND583201AAA' 'IND583101AAA'
 'IND583119AAA' 'IND602105AAB']


In [21]:
def build_features_v1(df):
    X = build_features(df).copy()

    X["source_center"] = df["source_center"].astype("category")
    X["destination_center"] = df["destination_center"].astype("category")

    return X

In [22]:
hub_categories = {
    "source_center": train["source_center"].unique(),
    "destination_center": train["destination_center"].unique()
}

def apply_hub_categories(X):
    X = X.copy()

    for col, categories in hub_categories.items():
        X[col] = pd.Categorical(
            X[col],
            categories=categories
        )

    return X

X_fit_v1 = apply_hub_categories(build_features_v1(fit_df))
X_val_v1 = apply_hub_categories(build_features_v1(val_df))
X_cal_v1 = apply_hub_categories(build_features_v1(cal))
X_test_v1 = apply_hub_categories(build_features_v1(test))

print("V1 feature shape:", X_fit_v1.shape)
print("V1 feature columns:", list(X_fit_v1.columns))
print("Unknown source hubs in validation:",
      X_val_v1["source_center"].isna().sum())
print("Unknown destination hubs in validation:",
      X_val_v1["destination_center"].isna().sum())

V1 feature shape: (12514, 10)
V1 feature columns: ['osrm_time_min', 'osrm_distance_km', 'osrm_speed_kmh', 'log_osrm_time', 'start_hour', 'start_dow', 'is_weekend', 'is_ftl', 'source_center', 'destination_center']
Unknown source hubs in validation: 0
Unknown destination hubs in validation: 0


In [23]:
xgb_v1 = XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    eval_metric="mae",
    early_stopping_rounds=50
)

xgb_v1.fit(
    X_fit_v1,
    y_fit,
    eval_set=[(X_val_v1, y_val)],
    verbose=False
)

print("Best iteration:", xgb_v1.best_iteration)
print("Best validation MAE:", xgb_v1.best_score)

Best iteration: 1970
Best validation MAE: 0.7820549652838883


In [24]:
v1_val_pred = xgb_v1.predict(X_val_v1)

v1_val_results = evaluate_model(
    y_val,
    v1_val_pred,
    val_df["osrm_time_min"]
)

print("V0 XGBoost:", v0_val_results)
print("V1 XGBoost:", v1_val_results)
print("Lane Median:", val_lane_results)

V0 XGBoost: {'MAE_ratio': 0.8981, 'MedAE_ratio': 0.361, 'MAE_excess_min': 45.29}
V1 XGBoost: {'MAE_ratio': 1.0344, 'MedAE_ratio': 0.2753, 'MAE_excess_min': 49.77}
Lane Median: {'MAE_ratio': 0.5662, 'MedAE_ratio': 0.1667, 'MAE_excess_min': 27.56}


In [25]:
v1_record = {
    "experiment": "V1",
    "description": "XGBoost with 8 engineered features and 2 hub categorical features",
    "features": list(X_fit_v1.columns),
    "fit_legs": len(fit_df),
    "validation_legs": len(val_df),
    "best_iteration": int(xgb_v1.best_iteration),
    "best_validation_mae": float(xgb_v1.best_score),
    "xgboost_validation_metrics": v1_val_results,
    "v0_validation_metrics": v0_val_results,
    "lane_median_validation_metrics": val_lane_results,
    "model_parameters": xgb_v1.get_params()
}

save_path = experiment_dir / "v1_results_v1.json"

if save_path.exists():
    raise FileExistsError(f"{save_path} already exists. Choose another version.")

with open(save_path, "w") as f:
    json.dump(v1_record, f, indent=4, default=str)

print("Saved:", save_path)

FileExistsError: ..\experiments\day2\v1_results_v1.json already exists. Choose another version.

In [26]:
lane_history = (
    fit_df.groupby(["source_center", "destination_center"])["delay_ratio"]
    .agg(["median", "count"])
    .reset_index()
)

lane_history.columns = [
    "source_center",
    "destination_center",
    "lane_median_ratio",
    "lane_history_count"
]

print("Historical lanes:", len(lane_history))
print("\nSample:")
display(lane_history.head())

Historical lanes: 2282

Sample:


,source_center,destination_center,lane_median_ratio,lane_history_count
0,IND000000AAL,IND411033AAA,2.541667,13
1,IND000000AAS,IND783370AAC,2.518519,4
2,IND000000AAZ,IND444203AAA,6.020833,1
3,IND000000ABA,IND683565AAA,1.000000,3
4,IND000000ABD,IND562132AAA,1.720270,8


In [27]:
def add_historical_lane_features(df, history_data=None, is_fit=False):
    result = df.copy()
    lane_cols = ["source_center", "destination_center"]

    if is_fit:
        result["lane_median_ratio"] = 1.0
        result["lane_history_count"] = 0

        for _, group in result.groupby(lane_cols, sort=False):
            group = group.sort_values("od_start_time")

            completed = group.sort_values("od_end_time")

            completed_delays = completed["delay_ratio"].to_numpy()

            completed_ends = pd.to_datetime(
              completed["od_end_time"],
              format="mixed"
             ).to_numpy(dtype="datetime64[ns]")

            pointer = 0
            past_delays = []

            for idx, row in group.iterrows():

                current_start = np.datetime64(
                    pd.Timestamp(row["od_start_time"]), "ns"
                )

                while (
                    pointer < len(completed)
                    and completed_ends[pointer] <= current_start
                ):
                    past_delays.append(completed_delays[pointer])
                    pointer += 1

                result.loc[idx, "lane_history_count"] = len(past_delays)

                if past_delays:
                    result.loc[idx, "lane_median_ratio"] = np.median(
                        past_delays
                    )

        return result

    if history_data is None:
        raise ValueError("Historical data is required.")

    lookup = history_data.set_index(lane_cols)
    keys = pd.MultiIndex.from_frame(result[lane_cols])

    result["lane_median_ratio"] = (
        lookup["lane_median_ratio"]
        .reindex(keys)
        .fillna(1.0)
        .to_numpy()
    )

    result["lane_history_count"] = (
        lookup["lane_history_count"]
        .reindex(keys)
        .fillna(0)
        .to_numpy()
    )

    return result

In [28]:
fit_v2 = add_historical_lane_features(
    fit_df,
    is_fit=True
)

val_v2 = add_historical_lane_features(
    val_df,
    history_data=lane_history
)

cal_v2 = add_historical_lane_features(
    cal,
    history_data=lane_history
)

test_v2 = add_historical_lane_features(
    test,
    history_data=lane_history
)

In [29]:
datasets = {
    "Fit": fit_v2,
    "Validation": val_v2,
    "Calibration": cal_v2,
    "Test": test_v2
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Missing lane medians:", df["lane_median_ratio"].isna().sum())
    print("Missing history counts:", df["lane_history_count"].isna().sum())
    print("Minimum history count:", df["lane_history_count"].min())
    print("Maximum history count:", df["lane_history_count"].max())
    print("Neutral fallback lanes:", (df["lane_history_count"] == 0).sum())


Fit
Shape: (12514, 16)
Missing lane medians: 0
Missing history counts: 0
Minimum history count: 0
Maximum history count: 62
Neutral fallback lanes: 2367

Validation
Shape: (3107, 16)
Missing lane medians: 0
Missing history counts: 0
Minimum history count: 0.0
Maximum history count: 65.0
Neutral fallback lanes: 145

Calibration
Shape: (5441, 16)
Missing lane medians: 0
Missing history counts: 0
Minimum history count: 0.0
Maximum history count: 65.0
Neutral fallback lanes: 345

Test
Shape: (5307, 16)
Missing lane medians: 0
Missing history counts: 0
Minimum history count: 0.0
Maximum history count: 65.0
Neutral fallback lanes: 538


In [30]:
print("Fit departure range:")
print(fit_df["od_start_time"].min(), "to", fit_df["od_start_time"].max())

print("\nValidation departure range:")
print(val_df["od_start_time"].min(), "to", val_df["od_start_time"].max())

print("\nFit completion range:")
print(fit_df["od_end_time"].min(), "to", fit_df["od_end_time"].max())

print("\nValidation departure before latest fit completion:")
print(
    (val_df["od_start_time"] <= fit_df["od_end_time"].max()).sum()
)

Fit departure range:
2018-09-12 00:00:16.535741 to 2018-09-23 12:42:32.876495

Validation departure range:
2018-09-21 21:46:36.360521 to 2018-09-26 03:52:15.492949

Fit completion range:
2018-09-12 00:50:10.814399 to 2018-09-24 14:42:59.483504

Validation departure before latest fit completion:
3005


In [31]:
for name, df in {
    "Fit": fit_df,
    "Validation": val_df,
    "Calibration": cal,
    "Test": test
}.items():
    print(
        name,
        df["od_start_time"].dtype,
        df["od_end_time"].dtype
    )

Fit datetime64[ns] object
Validation datetime64[ns] object
Calibration datetime64[ns] object
Test datetime64[ns] object


In [32]:
def add_historical_lane_features_asof(target_df, history_df):
    result = target_df.copy()

    lane_cols = ["source_center", "destination_center"]

    result["lane_median_ratio"] = 1.0
    result["lane_history_count"] = 0

    history_groups = {
        key: group.sort_values("od_end_time")
        for key, group in history_df.groupby(lane_cols, sort=False)
    }

    for key, target_group in result.groupby(lane_cols, sort=False):

        if key not in history_groups:
            continue

        hist = history_groups[key]

        end_times = hist["od_end_time"].to_numpy(
            dtype="datetime64[ns]"
        )
        delays = hist["delay_ratio"].to_numpy()

        for idx, row in target_group.iterrows():

            start_time = np.datetime64(
                row["od_start_time"], "ns"
            )

            count = np.searchsorted(
                end_times, start_time, side="right"
            )

            result.loc[idx, "lane_history_count"] = count

            if count > 0:
                result.loc[idx, "lane_median_ratio"] = np.median(
                    delays[:count]
                )

    return result

In [33]:
fit_v2 = add_historical_lane_features_asof(fit_df, fit_df)

val_v2 = add_historical_lane_features_asof(val_df, fit_df)

cal_v2 = add_historical_lane_features_asof(cal, train)

test_v2 = add_historical_lane_features_asof(test, train)

In [34]:
for name, df in {
    "Fit": fit_v2,
    "Validation": val_v2,
    "Calibration": cal_v2,
    "Test": test_v2
}.items():
    print(
        name,
        "| Shape:", df.shape,
        "| Missing:", df[["lane_median_ratio", "lane_history_count"]].isna().sum().sum(),
        "| Fallback rows:", (df["lane_history_count"] == 0).sum(),
        "| Max history:", df["lane_history_count"].max()
    )

Fit | Shape: (12514, 16) | Missing: 0 | Fallback rows: 2367 | Max history: 62
Validation | Shape: (3107, 16) | Missing: 0 | Fallback rows: 145 | Max history: 65
Calibration | Shape: (5441, 16) | Missing: 0 | Fallback rows: 240 | Max history: 85
Test | Shape: (5307, 16) | Missing: 0 | Fallback rows: 434 | Max history: 85


In [35]:
def build_features_v2(df):
    X = build_features_v1(df)

    X["lane_median_ratio"] = df["lane_median_ratio"]
    X["lane_history_count"] = df["lane_history_count"]

    return X

In [36]:
X_fit_v2 = apply_hub_categories(build_features_v2(fit_v2))
X_val_v2 = apply_hub_categories(build_features_v2(val_v2))
X_cal_v2 = apply_hub_categories(build_features_v2(cal_v2))
X_test_v2 = apply_hub_categories(build_features_v2(test_v2))

print("Fit:", X_fit_v2.shape)
print("Validation:", X_val_v2.shape)
print("Calibration:", X_cal_v2.shape)
print("Test:", X_test_v2.shape)
print("\nFeatures:")
print(X_fit_v2.columns.tolist())

Fit: (12514, 12)
Validation: (3107, 12)
Calibration: (5441, 12)
Test: (5307, 12)

Features:
['osrm_time_min', 'osrm_distance_km', 'osrm_speed_kmh', 'log_osrm_time', 'start_hour', 'start_dow', 'is_weekend', 'is_ftl', 'source_center', 'destination_center', 'lane_median_ratio', 'lane_history_count']


In [37]:
model_v2 = XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    eval_metric="mae",
    early_stopping_rounds=50
)

model_v2.fit(
    X_fit_v2,
    y_fit,
    eval_set=[(X_val_v2, y_val)],
    verbose=False
)

print("Best iteration:", model_v2.best_iteration)
print("Best validation MAE:", model_v2.best_score)

Best iteration: 1364
Best validation MAE: 0.767934828047051


In [38]:
pred_v2 = model_v2.predict(X_val_v2)

v2_metrics = evaluate_model(
    y_val,
    pred_v2,
    val_df["osrm_time_min"]
)

print("V2 Validation Metrics:")
print(v2_metrics)

V2 Validation Metrics:
{'MAE_ratio': 0.9531, 'MedAE_ratio': 0.2703, 'MAE_excess_min': 45.74}


In [40]:
print("Fit lane-history distribution:")
print(fit_v2["lane_history_count"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
))

print("\nValidation lane-history distribution:")
print(val_v2["lane_history_count"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
))

Fit lane-history distribution:
count    12514.000000
mean         5.139843
std          7.640881
min          0.000000
25%          1.000000
50%          3.000000
75%          6.000000
90%         12.000000
95%         19.000000
max         62.000000
Name: lane_history_count, dtype: float64

Validation lane-history distribution:
count    3107.000000
mean       10.557451
std        12.244931
min         0.000000
25%         4.000000
50%         7.000000
75%        10.000000
90%        23.000000
95%        39.000000
max        65.000000
Name: lane_history_count, dtype: float64


In [41]:
lane_history_metrics = evaluate_model(
    y_val,
    val_v2["lane_median_ratio"],
    val_df["osrm_time_min"]
)

print("Historical lane feature alone:")
print(lane_history_metrics)

print("\nLane history count correlation:")
print(
    val_v2[["lane_history_count", "delay_ratio"]]
    .corr(method="spearman")
)

Historical lane feature alone:
{'MAE_ratio': 0.7671, 'MedAE_ratio': 0.2584, 'MAE_excess_min': 37.84}

Lane history count correlation:
                    lane_history_count  delay_ratio
lane_history_count            1.000000    -0.025281
delay_ratio                  -0.025281     1.000000


In [42]:
print(
    lane_history["lane_history_count"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

count    2282.000000
mean        5.483786
std         5.752833
min         1.000000
25%         2.000000
50%         5.000000
75%         8.000000
90%         9.900000
95%        13.950000
max        65.000000
Name: lane_history_count, dtype: float64


In [43]:
lane_lookup = lane_history.set_index(
    ["source_center", "destination_center"]
)["lane_median_ratio"]

val_keys = pd.MultiIndex.from_frame(
    val_df[["source_center", "destination_center"]]
)

lane_median_val = (
    lane_lookup.reindex(val_keys)
    .fillna(global_median)
    .to_numpy()
)

print("Validation predictions:", len(lane_median_val))
print("Missing predictions:", pd.isna(lane_median_val).sum())

Validation predictions: 3107
Missing predictions: 0


In [44]:
comparison = pd.DataFrame({
    "original_lane_median": lane_median_val,
    "historical_lane_median": val_v2["lane_median_ratio"],
    "actual_delay_ratio": y_val
})

comparison["prediction_difference"] = (
    comparison["original_lane_median"]
    - comparison["historical_lane_median"]
).abs()

print(comparison.describe())
print(
    "\nPredictions with different values:",
    (comparison["prediction_difference"] > 0.000001).sum()
)

       original_lane_median  historical_lane_median  actual_delay_ratio  \
count           3107.000000             3107.000000         3107.000000   
mean               2.337364                2.290592            2.604644   
std                1.790825                1.811932            3.041654   
min                0.462114                0.462114            0.407895   
25%                1.724819                1.663179            1.675676   
50%                2.000000                1.971429            2.039604   
75%                2.449643                2.448605            2.596296   
max               39.866667               39.866667           77.387097   

       prediction_difference  
count            3107.000000  
mean                0.047120  
std                 0.210948  
min                 0.000000  
25%                 0.000000  
50%                 0.000000  
75%                 0.000000  
max                 1.000000  

Predictions with different values: 206


In [46]:
different = comparison[
    comparison["prediction_difference"] > 0.000001
].copy()

different["original_error"] = (
    different["original_lane_median"]
    - different["actual_delay_ratio"]
).abs()

different["historical_error"] = (
    different["historical_lane_median"]
    - different["actual_delay_ratio"]
).abs()

print(different[[
    "original_lane_median",
    "historical_lane_median",
    "actual_delay_ratio",
    "original_error",
    "historical_error"
]].head(10))

print("\nAverage original error:", different["original_error"].mean())
print("Average historical error:", different["historical_error"].mean())

       original_lane_median  historical_lane_median  actual_delay_ratio  \
12537              1.842105                1.815789            6.710526   
12539              2.099606                2.023124            2.021654   
12570              2.000000                1.000000            1.148936   
12575              2.559325                2.541163            2.556311   
12580              2.000000                1.000000            2.814815   
12593              2.226944                2.216998            2.158845   
12594              2.000000                1.000000            7.862069   
12602              2.101695                2.108225            1.280702   
12629              2.000000                1.000000            2.714286   
12630              2.000000                1.000000            1.619048   

       original_error  historical_error  
12537        4.868421          4.894737  
12539        0.077952          0.001470  
12570        0.851064          0.148936  
12575 

In [47]:
different_counts = val_v2.loc[
    different.index, "lane_history_count"
]

print("Differing predictions:", len(different))
print("Zero history:", (different_counts == 0).sum())
print("Some history:", (different_counts > 0).sum())

Differing predictions: 206
Zero history: 145
Some history: 61


In [48]:
zero_history_idx = val_v2.index[
    val_v2["lane_history_count"] == 0
]

original_fallback = lane_median_val[
    val_v2.index.get_indexer(zero_history_idx)
]

print("Zero-history validation legs:", len(zero_history_idx))
print(
    "Original lane median differs from 1.0:",
    (original_fallback != 1.0).sum()
)

Zero-history validation legs: 145
Original lane median differs from 1.0: 145


In [49]:
fit_v3 = fit_v2.copy()
val_v3 = val_v2.copy()

fit_v3["lane_median_ratio"] = np.where(
    fit_v3["lane_history_count"] == 0,
    fit_v3["lane_median_ratio"],
    fit_v3["lane_median_ratio"]
)

val_v3["lane_median_ratio"] = np.where(
    val_v3["lane_history_count"] == 0,
    lane_median_val,
    val_v3["lane_median_ratio"]
)

In [50]:
print("Zero-history rows before:", 
      (val_v2["lane_history_count"] == 0).sum())

print("Zero-history rows after:", 
      (val_v3["lane_history_count"] == 0).sum())

print("Fallback values updated:", 
      (val_v3["lane_median_ratio"] != val_v2["lane_median_ratio"]).sum())

Zero-history rows before: 145
Zero-history rows after: 145
Fallback values updated: 145


In [51]:
print("Unique source hubs:", fit_df["source_center"].nunique())
print("Unique source hubs in validation:", val_df["source_center"].nunique())

Unique source hubs: 1349
Unique source hubs in validation: 1008


In [52]:
source_history = (
    fit_df.groupby("source_center")["delay_ratio"]
    .agg(["median", "count"])
    .reset_index()
)

source_history.columns = [
    "source_center",
    "source_median_ratio",
    "source_history_count"
]

print("Source hubs:", len(source_history))
print(source_history.head())

Source hubs: 1349
  source_center  source_median_ratio  source_history_count
0  IND000000AAL             2.541667                    13
1  IND000000AAS             2.518519                     4
2  IND000000AAZ             6.020833                     1
3  IND000000ABA             1.000000                     3
4  IND000000ABD             1.720270                     8


In [53]:
source_history_asof = add_historical_lane_features_asof  # Keep existing lane function unchanged

print("Fit unique source hubs:", fit_df["source_center"].nunique())
print("Validation unique source hubs:", val_df["source_center"].nunique())

print(
    "Validation rows with zero lane history:",
    (val_v2["lane_history_count"] == 0).sum()
)

Fit unique source hubs: 1349
Validation unique source hubs: 1008
Validation rows with zero lane history: 145


In [54]:
source_groups = {
    key: group.sort_values("od_end_time")
    for key, group in fit_df.groupby("source_center", sort=False)
}

source_history_counts = []

for _, row in val_df.iterrows():
    hub = row["source_center"]

    if hub not in source_groups:
        source_history_counts.append(0)
        continue

    hist = source_groups[hub]
    end_times = hist["od_end_time"].to_numpy(dtype="datetime64[ns]")

    count = np.searchsorted(
        end_times,
        np.datetime64(row["od_start_time"], "ns"),
        side="right"
    )

    source_history_counts.append(count)

source_history_counts = np.array(source_history_counts)

print("Validation rows:", len(val_df))
print("Rows with source history:", (source_history_counts > 0).sum())
print("Rows without source history:", (source_history_counts == 0).sum())
print("Source history coverage:", round((source_history_counts > 0).mean() * 100, 2), "%")

Validation rows: 3107
Rows with source history: 3054
Rows without source history: 53
Source history coverage: 98.29 %


In [55]:
def add_source_history_asof(target_df, history_df):
    result = target_df.copy()
    result["source_median_ratio"] = 1.0
    result["source_history_count"] = 0

    history_groups = {
        key: group.sort_values("od_end_time")
        for key, group in history_df.groupby("source_center", sort=False)
    }

    for hub, target_group in result.groupby("source_center", sort=False):
        if hub not in history_groups:
            continue

        hist = history_groups[hub]

        end_times = hist["od_end_time"].to_numpy(
            dtype="datetime64[ns]"
        )
        delays = hist["delay_ratio"].to_numpy()

        for idx, row in target_group.iterrows():
            start_time = np.datetime64(row["od_start_time"], "ns")

            count = np.searchsorted(
                end_times,
                start_time,
                side="right"
            )

            result.loc[idx, "source_history_count"] = count

            if count > 0:
                result.loc[idx, "source_median_ratio"] = np.median(
                    delays[:count]
                )

    return result

In [56]:
fit_v3 = add_source_history_asof(fit_df, fit_df)

val_v3 = add_source_history_asof(val_df, fit_df)

cal_v3 = add_source_history_asof(cal, train)

test_v3 = add_source_history_asof(test, train)

In [57]:
for name, df in {
    "Fit": fit_v3,
    "Validation": val_v3,
    "Calibration": cal_v3,
    "Test": test_v3
}.items():

    zero_history = (df["source_history_count"] == 0).sum()

    print(f"\n{name}")
    print("Rows:", len(df))
    print("Zero source-history rows:", zero_history)
    print("Rows with source history:", len(df) - zero_history)
    print("Median source history:", df["source_history_count"].median())


Fit
Rows: 12514
Zero source-history rows: 1523
Rows with source history: 10991
Median source history: 6.0

Validation
Rows: 3107
Zero source-history rows: 53
Rows with source history: 3054
Median source history: 13.0

Calibration
Rows: 5441
Zero source-history rows: 86
Rows with source history: 5355
Median source history: 13.0

Test
Rows: 5307
Zero source-history rows: 168
Rows with source history: 5139
Median source history: 13.0


In [58]:
def combine_history_features(lane_df, source_df):
    result = lane_df.copy()

    use_source = result["lane_history_count"] == 0

    result["combined_history_ratio"] = np.where(
        use_source,
        source_df["source_median_ratio"],
        result["lane_median_ratio"]
    )

    result["combined_history_count"] = np.where(
        use_source,
        source_df["source_history_count"],
        result["lane_history_count"]
    )

    result["history_level"] = np.select(
        [
            result["lane_history_count"] > 0,
            source_df["source_history_count"] > 0
        ],
        ["lane", "source"],
        default="global"
    )

    return result


fit_v3 = combine_history_features(fit_v2, fit_v3)
val_v3 = combine_history_features(val_v2, val_v3)
cal_v3 = combine_history_features(cal_v2, cal_v3)
test_v3 = combine_history_features(test_v2, test_v3)

In [59]:
for name, df in {
    "Fit": fit_v3,
    "Validation": val_v3,
    "Calibration": cal_v3,
    "Test": test_v3
}.items():

    print(f"\n{name}")
    print(df["history_level"].value_counts())


Fit
history_level
lane      10147
global     1523
source      844
Name: count, dtype: int64

Validation
history_level
lane      2962
source      92
global      53
Name: count, dtype: int64

Calibration
history_level
lane      5201
source     154
global      86
Name: count, dtype: int64

Test
history_level
lane      4873
source     266
global     168
Name: count, dtype: int64


In [60]:
def build_features_v3(df):
    X = build_features_v1(df)

    X["combined_history_ratio"] = df["combined_history_ratio"]
    X["combined_history_count"] = df["combined_history_count"]

    return X


X_fit_v3 = build_features_v3(fit_v3)
X_val_v3 = build_features_v3(val_v3)
X_cal_v3 = build_features_v3(cal_v3)
X_test_v3 = build_features_v3(test_v3)

print("Fit:", X_fit_v3.shape)
print("Validation:", X_val_v3.shape)
print("Calibration:", X_cal_v3.shape)
print("Test:", X_test_v3.shape)

print("\nMissing values:")
print(X_fit_v3.isna().sum().sum(),
      X_val_v3.isna().sum().sum(),
      X_cal_v3.isna().sum().sum(),
      X_test_v3.isna().sum().sum())

Fit: (12514, 12)
Validation: (3107, 12)
Calibration: (5441, 12)
Test: (5307, 12)

Missing values:
0 0 0 0


In [61]:
from xgboost import XGBRegressor

xgb_v3 = XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    eval_metric="mae",
    early_stopping_rounds=50
)

xgb_v3.fit(
    X_fit_v3,
    y_fit,
    eval_set=[(X_val_v3, y_val)],
    verbose=False
)

print("Best iteration:", xgb_v3.best_iteration)
print("Best validation MAE:", xgb_v3.best_score)

Best iteration: 1035
Best validation MAE: 0.7745718375198474


In [62]:
pred_v3 = xgb_v3.predict(X_val_v3)

metrics_v3 = evaluate_model(
    y_val,
    pred_v3,
    val_df["osrm_time_min"]
)

print(metrics_v3)

{'MAE_ratio': 0.7746, 'MedAE_ratio': 0.2595, 'MAE_excess_min': 36.82}


In [63]:
import sklearn
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

print("Scikit-learn:", sklearn.__version__)
print("Random Forest: Ready")
print("Extra Trees: Ready")

Scikit-learn: 1.7.2
Random Forest: Ready
Extra Trees: Ready


In [64]:
from sklearn.preprocessing import OrdinalEncoder

categorical_cols = ["source_center", "destination_center"]

X_fit_v4 = X_fit_v2.copy()
X_val_v4 = X_val_v2.copy()

encoder_v4 = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_fit_v4[categorical_cols] = encoder_v4.fit_transform(
    X_fit_v4[categorical_cols].astype(str)
)

X_val_v4[categorical_cols] = encoder_v4.transform(
    X_val_v4[categorical_cols].astype(str)
)

print("Fit:", X_fit_v4.shape)
print("Validation:", X_val_v4.shape)

Fit: (12514, 12)
Validation: (3107, 12)


In [65]:
from sklearn.ensemble import RandomForestRegressor

rf_v4 = RandomForestRegressor(
    n_estimators=400,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.8,
    n_jobs=-1,
    random_state=42
)

rf_v4.fit(X_fit_v4, y_fit)

print("Random Forest training completed!")

Random Forest training completed!


In [66]:
pred_v4 = rf_v4.predict(X_val_v4)

metrics_v4 = evaluate_model(
    y_val,
    pred_v4,
    val_df["osrm_time_min"]
)

print("V4 Random Forest:", metrics_v4)

V4 Random Forest: {'MAE_ratio': 0.8106, 'MedAE_ratio': 0.3387, 'MAE_excess_min': 38.65}


In [67]:
model_comparison = pd.DataFrame([
    {"Model": "V0 - Basic XGBoost", "MAE": 0.8981},
    {"Model": "V1 - Hub XGBoost", "MAE": 1.0344},
    {"Model": "V2 - Historical Lane XGBoost", "MAE": 0.7679},
    {"Model": "V3 - Lane + Source XGBoost", "MAE": 0.7746},
    {"Model": "V4 - Random Forest", "MAE": metrics_v4["MAE_ratio"]},
    {"Model": "Lane Median Baseline", "MAE": 0.5662}
])

model_comparison = model_comparison.sort_values("MAE")

display(model_comparison)

,Model,MAE
5,Lane Median Baseline,0.5662
2,V2 - Historical Lane XGBoost,0.7679
3,V3 - Lane + Source XGBoost,0.7746
4,V4 - Random Forest,0.8106
0,V0 - Basic XGBoost,0.8981
1,V1 - Hub XGBoost,1.0344


In [68]:
import json
from pathlib import Path

output_dir = Path("../experiments/day2")
output_dir.mkdir(parents=True, exist_ok=True)

result_v4 = {
    "model": "Random Forest Regressor",
    "features": "V2 historical lane features",
    "parameters": rf_v4.get_params(),
    "validation_metrics": metrics_v4
}

output_path = output_dir / "v4_random_forest_results_v1.json"

if output_path.exists():
    raise FileExistsError(f"{output_path} already exists. Choose a new version.")

with open(output_path, "w") as f:
    json.dump(result_v4, f, indent=4, default=str)

print("Saved:", output_path)

FileExistsError: ..\experiments\day2\v4_random_forest_results_v1.json already exists. Choose a new version.

In [69]:
from sklearn.ensemble import ExtraTreesRegressor

et_v5 = ExtraTreesRegressor(
    n_estimators=400,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.8,
    n_jobs=-1,
    random_state=42
)

et_v5.fit(X_fit_v4, y_fit)

print("Extra Trees training completed!")

Extra Trees training completed!


In [70]:
pred_v5 = et_v5.predict(X_val_v4)

metrics_v5 = evaluate_model(
    y_val,
    pred_v5,
    val_df["osrm_time_min"]
)

print("V5 Extra Trees:", metrics_v5)

V5 Extra Trees: {'MAE_ratio': 0.7812, 'MedAE_ratio': 0.3303, 'MAE_excess_min': 36.78}


In [71]:
result_v5 = {
    "model": "Extra Trees Regressor",
    "features": "V2 historical lane features",
    "parameters": et_v5.get_params(),
    "validation_metrics": metrics_v5
}

output_path = output_dir / "v5_extra_trees_results_v1.json"

if output_path.exists():
    raise FileExistsError(f"{output_path} already exists.")

with open(output_path, "w") as f:
    json.dump(result_v5, f, indent=4, default=str)

print("Saved:", output_path)

FileExistsError: ..\experiments\day2\v5_extra_trees_results_v1.json already exists.

In [72]:
fair_lane_val = evaluate_model(y_val, lane_median_val, val_df["osrm_time_min"])
print("Fair lane baseline on validation:", fair_lane_val)

Fair lane baseline on validation: {'MAE_ratio': 0.7366, 'MedAE_ratio': 0.25, 'MAE_excess_min': 35.58}


In [73]:
rows = {
    "Lane median (fit-only, fair)": lane_median_val,
    "V0 basic":          xgb_v0.predict(X_val),
    "V1 hubs":           xgb_v1.predict(X_val_v1),
    "V2 lane history":   model_v2.predict(X_val_v2),
    "V3 lane + source":  xgb_v3.predict(X_val_v3),
    "V4 random forest":  rf_v4.predict(X_val_v4),
}
table = pd.DataFrame({
    k: evaluate_model(y_val, p, val_df["osrm_time_min"]) for k, p in rows.items()
}).T
table.sort_values("MAE_ratio")

,MAE_ratio,MedAE_ratio,MAE_excess_min
"Lane median (fit-only, fair)",0.7366,0.2500,35.58
V3 lane + source,0.7746,0.2595,36.82
V4 random forest,0.8106,0.3387,38.65
V0 basic,0.8981,0.3610,45.29
V2 lane history,0.9531,0.2703,45.74
V1 hubs,1.0344,0.2753,49.77


In [74]:
from sklearn.metrics import mean_absolute_error
for name, m, X in [("V1", xgb_v1, X_val_v1), ("V2", model_v2, X_val_v2)]:
    p = m.predict(X)
    print(name, "recomputed:", round(mean_absolute_error(y_val, p), 4),
          "| XGBoost best_score:", round(m.best_score, 4))

V1 recomputed: 1.0344 | XGBoost best_score: 0.7821
V2 recomputed: 0.9531 | XGBoost best_score: 0.7679


In [75]:
g_fit = float(fit_df["delay_ratio"].median())

def base_pred(d):
    # lane median if we have history, else global median
    return d["lane_median_ratio"].where(d["lane_history_count"] > 0, g_fit).to_numpy()

num_cols = ["osrm_time_min", "osrm_distance_km", "osrm_speed_kmh", "log_osrm_time",
            "start_hour", "start_dow", "is_weekend", "is_ftl"]

def make_X(d):
    X = build_features(d)[num_cols].copy()
    X["lane_base"] = base_pred(d)
    X["lane_history_count"] = d["lane_history_count"].to_numpy()
    for c in ["recent_hub_median", "recent_hub_count"]:   # used in Step 4
        if c in d.columns:
            X[c] = d[c].to_numpy()
    return X

base_fit, base_val = base_pred(fit_v2), base_pred(val_v2)
Xf, Xv = make_X(fit_v2), make_X(val_v2)

res_model = XGBRegressor(
    objective="reg:absoluteerror", n_estimators=3000, learning_rate=0.03,
    max_depth=3, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=5, tree_method="hist", random_state=42,
    eval_metric="mae", early_stopping_rounds=100)

res_model.fit(Xf, y_fit.to_numpy() - base_fit,
              eval_set=[(Xv, y_val.to_numpy() - base_val)], verbose=False)

pred_val = base_val + res_model.predict(Xv)
print("Base alone     :", evaluate_model(y_val, base_val, val_df["osrm_time_min"]))

print("Base + XGBoost :", evaluate_model(y_val, pred_val, val_df["osrm_time_min"]))

Base alone     : {'MAE_ratio': 0.7366, 'MedAE_ratio': 0.25, 'MAE_excess_min': 35.54}
Base + XGBoost : {'MAE_ratio': 0.7141, 'MedAE_ratio': 0.2459, 'MAE_excess_min': 34.39}


In [76]:
def add_recent_hub_delay(target_df, history_df, window_hours=48):
    hist = history_df[["source_center", "od_end_time", "delay_ratio"]].copy()
    hist["end_ts"] = pd.to_datetime(hist["od_end_time"], format="mixed")
    hist = hist.sort_values("end_ts")
    groups = {hub: (g["end_ts"].to_numpy("datetime64[ns]"), g["delay_ratio"].to_numpy())
              for hub, g in hist.groupby("source_center")}
    window = np.timedelta64(window_hours, "h")
    starts = pd.to_datetime(target_df["od_start_time"]).to_numpy("datetime64[ns]")
    hubs = target_df["source_center"].to_numpy()

    med = np.full(len(target_df), np.nan)
    cnt = np.zeros(len(target_df), dtype=int)
    for i, (hub, s) in enumerate(zip(hubs, starts)):
        if hub not in groups:
            continue
        ends, delays = groups[hub]
        hi = np.searchsorted(ends, s, side="right")           # finished before departure
        lo = np.searchsorted(ends, s - window, side="left")   # within the last 48 hours
        if hi > lo:
            med[i] = np.median(delays[lo:hi])
            cnt[i] = hi - lo
    out = target_df.copy()
    out["recent_hub_median"] = med
    out["recent_hub_count"] = cnt
    return out

In [77]:
fit_r  = add_recent_hub_delay(fit_v2,  fit_df)
val_r  = add_recent_hub_delay(val_v2,  train)
cal_r  = add_recent_hub_delay(cal_v2,  pd.concat([train, cal]))
test_r = add_recent_hub_delay(test_v2, pd.concat([train, cal, test]))

print("Share of rows with recent hub history:",
      {n: round((d["recent_hub_count"] > 0).mean(), 3)
       for n, d in [("fit", fit_r), ("val", val_r), ("cal", cal_r), ("test", test_r)]})

Share of rows with recent hub history: {'fit': np.float64(0.831), 'val': np.float64(0.91), 'cal': np.float64(0.899), 'test': np.float64(0.905)}


In [78]:
from sklearn.metrics import mean_absolute_error

def run_residual(fit_frame, val_frame, seed=42):
    b_fit, b_val = base_pred(fit_frame), base_pred(val_frame)
    Xf, Xv = make_X(fit_frame), make_X(val_frame)
    m = XGBRegressor(
        objective="reg:absoluteerror", n_estimators=3000, learning_rate=0.03,
        max_depth=3, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=5, tree_method="hist", random_state=seed,
        eval_metric="mae", early_stopping_rounds=100)
    m.fit(Xf, y_fit.to_numpy() - b_fit,
          eval_set=[(Xv, y_val.to_numpy() - b_val)], verbose=False)
    pred = b_val + m.predict(Xv)
    return m, pred, b_val

m_plain,  pred_plain,  base_val = run_residual(fit_v2, val_v2)   # no recent features
m_recent, pred_recent, _        = run_residual(fit_r,  val_r)    # with recent hub delay

# sanity check: XGBoost's own score should now match yours
print("best_score (plain) :", round(m_plain.best_score, 4),
      "| recomputed:", round(mean_absolute_error(y_val, pred_plain), 4))
print("best_score (recent):", round(m_recent.best_score, 4),
      "| recomputed:", round(mean_absolute_error(y_val, pred_recent), 4))

ablation = pd.DataFrame({
    "Base alone (lane median)":            evaluate_model(y_val, base_val,    val_df["osrm_time_min"]),
    "Base + XGBoost (no recent)":          evaluate_model(y_val, pred_plain,  val_df["osrm_time_min"]),
    "Base + XGBoost (+ recent hub delay)": evaluate_model(y_val, pred_recent, val_df["osrm_time_min"]),
}).T
ablation

best_score (plain) : 0.7141 | recomputed: 0.7141
best_score (recent): 0.7125 | recomputed: 0.7125


,MAE_ratio,MedAE_ratio,MAE_excess_min
Base alone (lane median),0.7366,0.2500,35.54
Base + XGBoost (no recent),0.7141,0.2459,34.39
Base + XGBoost (+ recent hub delay),0.7125,0.2403,34.37


In [79]:
def paired_trip_bootstrap(y, p_new, p_base, trips, n=2000, seed=0):
    d = np.abs(y - p_base) - np.abs(y - p_new)      # positive = new is better
    g = pd.DataFrame({"trip": trips, "d": d}).groupby("trip")["d"].agg(["sum", "count"])
    s, c = g["sum"].to_numpy(), g["count"].to_numpy()
    rng = np.random.default_rng(seed)
    boots = [s[i].sum() / c[i].sum()
             for i in (rng.integers(0, len(g), len(g)) for _ in range(n))]
    return np.percentile(boots, [2.5, 50, 97.5])

y = y_val.to_numpy()
trips = val_df["trip_uuid"].to_numpy()
print("plain  vs base:", paired_trip_bootstrap(y, pred_plain,  base_val, trips))
print("recent vs base:", paired_trip_bootstrap(y, pred_recent, base_val, trips))

plain  vs base: [0.01109417 0.02236796 0.03389325]
recent vs base: [0.01025205 0.02397412 0.03819377]


In [80]:
val_start = val_df["od_start_time"].min()
late = pd.to_datetime(fit_df["od_end_time"], format="mixed") > val_start
print("Fit legs that finish after validation begins:", late.sum(), "of", len(fit_df))

Fit legs that finish after validation begins: 576 of 12514


In [81]:
def late_share(earlier, later, name):
    ends = pd.to_datetime(earlier["od_end_time"], format="mixed")
    boundary = pd.to_datetime(later["od_start_time"]).min()
    n = int((ends > boundary).sum())
    print(f"{name}: {n} of {len(earlier)} legs ({n/len(earlier):.1%}) finish after "
          f"the next pile's first departure ({boundary})")

late_share(fit_df, val_df, "fit -> val")
late_share(train, cal, "train -> cal")
late_share(pd.concat([train, cal]), test, "train+cal -> test")

fit -> val: 576 of 12514 legs (4.6%) finish after the next pile's first departure (2018-09-21 21:46:36.360521)
train -> cal: 324 of 15621 legs (2.1%) finish after the next pile's first departure (2018-09-24 13:23:41.674827)
train+cal -> test: 520 of 21062 legs (2.5%) finish after the next pile's first departure (2018-09-28 22:36:42.571355)


In [82]:
print("recent vs plain:", paired_trip_bootstrap(y, pred_recent, pred_plain, trips))

recent vs plain: [-0.00171881  0.00166057  0.00500853]


In [83]:
def make_X(d):
    X = build_features(d)[num_cols].copy()
    X["lane_base"] = base_pred(d)
    X["lane_history_count"] = d["lane_history_count"].to_numpy()
    for c in ["recent_hub_median", "recent_hub_count", "prev_delay_ratio", "prev_connected"]:
        if c in d.columns:
            X[c] = d[c].to_numpy()
    return X

def add_prev_leg_features(d):
    out = d.copy()
    out["_start"] = pd.to_datetime(out["od_start_time"])
    out["_end"] = pd.to_datetime(out["od_end_time"], format="mixed")
    out = out.sort_values(["trip_uuid", "_start"])
    g = out.groupby("trip_uuid")
    prev_end   = g["_end"].shift(1)
    prev_ratio = g["delay_ratio"].shift(1)
    prev_dest  = g["destination_center"].shift(1)
    ok = prev_end.notna() & (prev_end <= out["_start"]) # previous leg already finished
    out["prev_delay_ratio"] = prev_ratio.where(ok)
    out["prev_connected"] = ((prev_dest == out["source_center"]) & ok).astype(int)
    return out.drop(columns=["_start", "_end"]).loc[d.index]

fit_p, val_p = add_prev_leg_features(fit_r), add_prev_leg_features(val_r)
cal_p, test_p = add_prev_leg_features(cal_r), add_prev_leg_features(test_r)

print("Rows with a finished previous leg:", {n: round(d["prev_delay_ratio"].notna().mean(), 3) for n, d in [("fit", fit_p), ("val", val_p), ("cal", cal_p), ("test", test_p)]})

m_prev, pred_prev, _ = run_residual(fit_p, val_p)
print("best_score:", round(m_prev.best_score, 4), "| recomputed:", round(mean_absolute_error(y_val, pred_prev), 4))

ablation2 = pd.DataFrame({
    "Base alone": evaluate_model(y_val, base_val, val_df["osrm_time_min"]),
    "XGBoost (no recent)": evaluate_model(y_val, pred_plain, val_df["osrm_time_min"]),
    "XGBoost (+ recent hub)": evaluate_model(y_val, pred_recent, val_df["osrm_time_min"]),
    "XGBoost (+ recent + prev leg)": evaluate_model(y_val, pred_prev, val_df["osrm_time_min"]),
}).T
print(ablation2)
print("prev vs recent:", paired_trip_bootstrap(y, pred_prev, pred_recent, trips))

Rows with a finished previous leg: {'fit': np.float64(0.432), 'val': np.float64(0.428), 'cal': np.float64(0.455), 'test': np.float64(0.441)}
best_score: 0.7136 | recomputed: 0.7136
                               MAE_ratio  MedAE_ratio  MAE_excess_min
Base alone                        0.7366       0.2500           35.54
XGBoost (no recent)               0.7141       0.2459           34.39
XGBoost (+ recent hub)            0.7125       0.2403           34.37
XGBoost (+ recent + prev leg)     0.7136       0.2403           34.34
prev vs recent: [-0.00426497 -0.00110323  0.0022    ]


In [84]:
def final_eval(model, cal_frame, test_frame):
    out = {}
    for name, frame, y_true in [("cal", cal_frame, cal["delay_ratio"]),
                                ("test", test_frame, test["delay_ratio"])]:
        base = base_pred(frame)
        pred = base + model.predict(make_X(frame))
        out[f"{name}: base alone"]      = evaluate_model(y_true, base, frame["osrm_time_min"])
        out[f"{name}: base + XGBoost"]  = evaluate_model(y_true, pred, frame["osrm_time_min"])
    return pd.DataFrame(out).T

final_eval(m_prev, cal_p, test_p) # use m_recent / m_plain with their frames if they won

m_prev.save_model("../experiments/day2/residual_xgb_v1_check.json")



In [85]:
import pandas as pd

enr = pd.read_csv("../data/processed/legs_enriched.csv")
legs = pd.read_csv("../data/processed/legs.csv")

print("Shape:", enr.shape, "| legs.csv shape:", legs.shape)
print("Same number of rows:", len(enr) == len(legs))

new_cols = [c for c in enr.columns if c not in legs.columns]
print("\nNew columns added by Person A:", new_cols)

keys = [c for c in ["trip_uuid", "source_center", "destination_center", "od_start_time"] if c in enr.columns]
print("Join keys available:", keys)
print("Duplicate keys:", enr.duplicated(subset=keys).sum())
print("\nShare missing per new column:")
print(enr[new_cols].isna().mean().round(3))
print("\nSummary of new columns:")
print(enr[new_cols].describe().T)

Shape: (7817, 27) | legs.csv shape: (26369, 14)
Same number of rows: False

New columns added by Person A: ['trip_created', 'split', 'departure_date', 'arrival_date', 'source_rain_mm', 'source_precipitation_mm', 'source_temperature_mean_c', 'source_weather_available', 'destination_rain_mm', 'destination_precipitation_mm', 'destination_temperature_mean_c', 'destination_weather_available', 'both_weather_available']
Join keys available: ['trip_uuid', 'source_center', 'destination_center', 'od_start_time']
Duplicate keys: 0

Share missing per new column:
trip_created                      0.000
split                             0.000
departure_date                    0.000
arrival_date                      0.000
source_rain_mm                    0.037
source_precipitation_mm           0.037
source_temperature_mean_c         0.037
source_weather_available          0.000
destination_rain_mm               0.048
destination_precipitation_mm      0.048
destination_temperature_mean_c    0.048
des

In [86]:
def final_eval(model, cal_frame, test_frame):
    out = {}
    for name, frame, y_true in [("cal", cal_frame, cal["delay_ratio"]), ("test", test_frame, test["delay_ratio"])]:
        base = base_pred(frame)
        pred = base + model.predict(make_X(frame))
        out[f"{name} : base alone"] = evaluate_model(y_true, base, frame["osrm_time_min"])
        out[f"{name} : base + XGBoost"] = evaluate_model(y_true, pred, frame["osrm_time_min"])
    return pd.DataFrame(out).T

# Using m_recent configuration since it achieved the best MAE ratio (0.7125)
print(final_eval(m_recent, cal_r, test_r))

                       MAE_ratio  MedAE_ratio  MAE_excess_min
cal : base alone          0.6395       0.2478           29.37
cal : base + XGBoost      0.6287       0.2388           29.51
test : base alone         0.7406       0.2851           34.27
test : base + XGBoost     0.7151       0.2756           33.51


In [87]:
import json
from pathlib import Path

# Ensure directory exists
Path("../experiments/day2").mkdir(parents=True, exist_ok=True)

# Save evaluation results dictionary
final_results_dict = {
    "calibration_base_alone_mae_ratio": 0.6395,
    "calibration_xgboost_mae_ratio": 0.6287,
    "test_base_alone_mae_ratio": 0.7406,
    "test_xgboost_mae_ratio": 0.7151
}

with open("../experiments/day2/final_eval_v1.json", "w") as f:
    json.dump(final_results_dict, f, indent=4)

# Save your winning residual XGBoost model
m_recent.save_model("../experiments/day2/residual_xgb_v1.json")
print("Successfully saved final evaluation JSON and XGBoost model!")

Successfully saved final evaluation JSON and XGBoost model!


In [88]:
enr = pd.read_csv("../data/processed/legs_enriched.csv")
mine = pd.concat([train.assign(my_split="train"), cal.assign(my_split="cal"), test.assign(my_split="test")])

k = ["trip_uuid", "source_center", "destination_center"]
m = enr.merge(mine[k + ["my_split"]], on=k, how="left")

print("Legs in weather file:", len(enr), "| hubs:",
      len(set(enr.source_center) | set(enr.destination_center)))
print("Matched to your piles:", m["my_split"].notna().sum(), "of", len(enr))
print(pd.crosstab(m["split"], m["my_split"]))

Legs in weather file: 7817 | hubs: 100
Matched to your piles: 7817 of 7817
my_split  cal  test  train
split                     
later     950  1472      0
training  553     0   4842


In [89]:
final_model = m_plain
res = final_eval(final_model, cal_v2, test_v2)
print(res)

final_model.save_model("../models/residual_xgb_v1.json")
res.to_json("../experiments/day2/final_eval_v1.json", indent=2)

                       MAE_ratio  MedAE_ratio  MAE_excess_min
cal : base alone          0.6395       0.2478           29.37
cal : base + XGBoost      0.6288       0.2352           29.35
test : base alone         0.7406       0.2851           34.27
test : base + XGBoost     0.7195       0.2806           33.57


In [90]:
import shutil, json

shutil.move("../models/residual_xgb_v1.json",
            "../models/residual_xgb_plain_v1.json")                 # FINAL model
shutil.move("../experiments/day2/residual_xgb_v1.json",
            "../experiments/day2/candidate_residual_xgb_recent_v1.json")   # rejected candidate

card = {
    "final_model": "residual XGBoost, plain (lane median + learned correction)",
    "file": "models/residual_xgb_plain_v1.json",
    "features": list(final_model.feature_names_in_),
    "best_iteration": int(final_model.best_iteration),
    "trained_on": "fit pile (first 80% of train trips); early-stopped on validation pile (last 20%)",
    "results": res.round(4).to_dict(orient="index"),
    "notes": [
        "recent-hub-delay and previous-leg features tested: no significant gain (bootstrap ranges include 0)",
        "hub-ID categorical models dropped: XGBoost score disagreed with recomputed MAE",
        "embargo overlap: 2.1% to 4.6% of legs finish after the next pile starts (limitation)",
    ],
}
with open("../experiments/day2/final_model_card_v1.json", "w") as f:
    json.dump(card, f, indent=2)

In [91]:
cols = ["trip_uuid", "source_center", "destination_center", "route_type", "od_start_time",
        "osrm_time_min", "osrm_distance_km", "lane_history_count", "delay_ratio"]

def export_preds(model, frame, pile):
    out = frame[cols].copy()
    out["base_pred"]  = base_pred(frame)
    out["pred_ratio"] = out["base_pred"] + model.predict(make_X(frame))
    out["pile"] = pile
    return out.rename(columns={"delay_ratio": "y_ratio"})

preds = pd.concat([export_preds(final_model, cal_v2, "cal"),
                   export_preds(final_model, test_v2, "test")], ignore_index=True)

# sanity check: must reproduce your official numbers
for pile in ["cal", "test"]:
    d = preds[preds["pile"] == pile]
    print(pile, "MAE:", round((d["y_ratio"] - d["pred_ratio"]).abs().mean(), 4), "| rows:", len(d))

preds.to_csv("../data/processed/preds_cal_test_v1.csv", index=False)
preds[preds["pile"] == "test"].to_csv("../data/processed/preds_test.csv", index=False)

cal MAE: 0.6288 | rows: 5441
test MAE: 0.7195 | rows: 5307
